# Sentetik Veri Üretimi ve Adversarial Filtering

Bu çalışma dosyasının temel amacı, yüksek şiddetli orman yangınlarının (Mega Fires) nadir görülmesinden kaynaklanan veri dengesizliği (class imbalance) problemini çözmektir. Klasik aşırı örnekleme (Oversampling) tekniklerinin yetersiz kaldığı bu çok boyutlu veri setinde, özellikler arası yapısal korelasyonları (Joint Distribution) ve ekstrem uç değerleri (Tail Distributions) koruyabilen jeneratif istatistiksel modeller kullanılmıştır. Üretilen verilerin kalitesi, döngüsel değerlendirme (Circular Evaluation) tuzaklarına düşülmeden, bağımsız istatistiksel testler ile doğrulanmıştır.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.covariance import LedoitWolf
from copulas.multivariate import GaussianMultivariate
from ctgan import CTGAN

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_predict
from scipy.stats import anderson_ksamp

## 1. Hedef Kümenin (Mega Yangınlar) İzole Edilmesi
Sentetik verinin amacı tüm veri setini değil, modelin öğrenmekte zorlandığı "yüksek hasarlı yangın" örüntülerini çoğaltmaktır. Bu doğrultuda, sıfırdan büyük alana sahip yangınların en üst %25'lik çeyreği (Top Quartile) izole edilerek "Elit Gözlem Kümesi" olarak tanımlanmıştır. Üretim modelleri sadece bu hedef kitlenin karakteristiğini öğrenecektir.

In [2]:
X_train = pd.read_csv('../../data/processed/X_train_v2.csv')
y_train = pd.read_csv('../../data/processed/y_train_v2.csv')

df_train = pd.concat([X_train, y_train], axis=1)

print(f"Toplam Gözlem Sayısı (N): {len(df_train)}")

pozitif_yanginlar = df_train[df_train['area'] > 0].copy()
print(f"Pozitif Hasarlı Yangın Sayısı: {len(pozitif_yanginlar)}")

q75_siniri = pozitif_yanginlar['area'].quantile(0.75)
elit_mega_yanginlar = pozitif_yanginlar[pozitif_yanginlar['area'] >= q75_siniri].copy()

print(f"Top Quartile Sınırı (Log Area): {q75_siniri:.4f}")
print(f"Laboratuvara Girecek Elit Gözlem Sayısı (N): {len(elit_mega_yanginlar)}")

Toplam Gözlem Sayısı (N): 413
Pozitif Hasarlı Yangın Sayısı: 215
Top Quartile Sınırı (Log Area): 2.7970
Laboratuvara Girecek Elit Gözlem Sayısı (N): 54


## 2. Boyut İndirgeme ve Fiziksel Gerçeklik Sınırlarının Belirlenmesi
Jeneratif modellerin kısıtlı gözlem sayısıyla (n=54) çok boyutlu uzayda (Curse of Dimensionality) başarısız olmasını engellemek adına, döngüsel (cyclical) ve nedensel ağırlığı düşük olan değişkenler çıkarılarak "Çekirdek Değişken Uzayı" kurgulanmıştır.

Üretilen sentetik verilerin fizik kanunlarına ve parkın coğrafi gerçeklerine aykırı olmaması (örn: negatif nem veya 150°C sıcaklık) için, orijinal eğitim setindeki minimum ve maksimum değerler referans alınarak **Fiziksel Kırpma (Physical Clipping)** algoritması tanımlanmıştır.

In [3]:
cekirdek_kolonlar = [
    'X', 'Y', 'FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 
    'month_sin', 'month_cos', 'area'
]

hedef_veri = elit_mega_yanginlar[cekirdek_kolonlar].astype(float)

D = len(cekirdek_kolonlar)
N = len(hedef_veri)

print("--- BOYUT İNDİRGEME (DIMENSIONALITY REDUCTION) ---")
print(f"Orijinal D (Boyut) Sayısı: {len(df_train.columns)}")
print(f"Çekirdek D (Boyut) Sayısı: {D}")
print(f"Gözlem Sayısı (N): {N}")

--- BOYUT İNDİRGEME (DIMENSIONALITY REDUCTION) ---
Orijinal D (Boyut) Sayısı: 23
Çekirdek D (Boyut) Sayısı: 12
Gözlem Sayısı (N): 54


In [4]:
fiziksel_sinirlar = {}

for col in cekirdek_kolonlar:
    fiziksel_sinirlar[col] = {
        'min': df_train[col].min(),
        'max': df_train[col].max()
    }

print(f"RH (Nem) Sınırları     : {fiziksel_sinirlar['RH']['min']} - {fiziksel_sinirlar['RH']['max']}")
print(f"Temp (Sıcaklık) Sınırları: {fiziksel_sinirlar['temp']['min']} - {fiziksel_sinirlar['temp']['max']}")
print(f"Month_Sin (Mevsim) Sınırı: {fiziksel_sinirlar['month_sin']['min']} - {fiziksel_sinirlar['month_sin']['max']}")

def fiziksel_sinirlara_hapset(df_sentetik):
    df_clipped = df_sentetik.copy()
    for col in cekirdek_kolonlar:
        df_clipped[col] = df_clipped[col].clip(
            lower=fiziksel_sinirlar[col]['min'], 
            upper=fiziksel_sinirlar[col]['max']
        )
    df_clipped['X'] = df_clipped['X'].round()
    df_clipped['Y'] = df_clipped['Y'].round()
    return df_clipped

RH (Nem) Sınırları     : 15 - 100
Temp (Sıcaklık) Sınırları: 2.2 - 33.1
Month_Sin (Mevsim) Sınırı: -1.0 - 1.0


## 3. Sentetik Veri Üretim Adaylarının (Generative Models) Eğitilmesi
Veri artırımı için üç farklı istatistiksel ve makine öğrenmesi tabanlı mimari yarışmaktadır:
* **Aday A (Ledoit-Wolf Gaussian):** Düşük gözlem sayısında kovaryans matrisi kestirimini düzenlileştiren (regularization) geleneksel çok değişkenli normal dağılım yaklaşımı.
* **Aday B (Gaussian Copula):** Değişkenlerin kendi içlerindeki (marginal) dağılımları ile birbirleri arasındaki bağımlılık yapılarını ayırarak modelleyen kopula mimarisi.
* **Aday C (CTGAN):** Derin öğrenme tabanlı Koşullu Tablo Üretici Çekişmeli Ağ (Conditional Tabular GAN) mimarisi.

In [5]:
lw = LedoitWolf()
lw.fit(hedef_veri)

mu_lw = hedef_veri.mean().values
sigma_lw = lw.covariance_

np.random.seed(42)
matris_A = np.random.multivariate_normal(mean=mu_lw, cov=sigma_lw, size=100)
df_sentetik_A = pd.DataFrame(matris_A, columns=cekirdek_kolonlar)

df_sentetik_A = fiziksel_sinirlara_hapset(df_sentetik_A)

In [6]:
copula_model = GaussianMultivariate()
copula_model.fit(hedef_veri)

df_sentetik_B = copula_model.sample(100)
df_sentetik_B = fiziksel_sinirlara_hapset(df_sentetik_B)

In [7]:
ctgan_model = CTGAN(epochs=100, verbose=False)
ctgan_model.fit(hedef_veri)

df_sentetik_C = ctgan_model.sample(100)
df_sentetik_C = fiziksel_sinirlara_hapset(df_sentetik_C)

## 4. Ham (Filtresiz) Adayların Yapısal Testleri
Aday modellerin ürettiği veriler ilk aşamada filtrelenmeden iki temel teste tabi tutulmuştur:
1. **Adversarial Validation (Düşman Doğrulama):** Sentetik ve gerçek veriyi ayırt etmeye çalışan bir sınıflandırıcı (Random Forest) üzerinden AUC ölçümü. Skorun 0.50'ye yakınsaması verinin gerçeğe benzerliğini gösterir.
2. **Frobenius Normu:** Orijinal verinin korelasyon matrisi ile sentetik verinin matrisi arasındaki uzaklığı (Fiziksel Yapı Hatalarını) ölçer.

In [8]:
def adversarial_validation(real_data, synthetic_data, name):
    syn_sample = synthetic_data.head(len(real_data))
    X_adv = pd.concat([real_data, syn_sample]).reset_index(drop=True)
    y_adv = np.array([0] * len(real_data) + [1] * len(syn_sample))
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
    preds = cross_val_predict(rf, X_adv, y_adv, cv=5, method='predict_proba')[:, 1]
    
    auc = roc_auc_score(y_adv, preds)
    hata_payi = abs(auc - 0.50)
    
    print(f"{name:.<40} AUC Skoru: {auc:.4f} | Mükemmelliğe Uzaklık: {hata_payi:.4f}")
    return hata_payi

def frobenius_norm_test(real_data, synthetic_data, name):
    syn_sample = synthetic_data.head(len(real_data))
    corr_real = np.nan_to_num(real_data.corr().values, nan=0.0)
    corr_synth = np.nan_to_num(syn_sample.corr().values, nan=0.0)
    
    fark_normu = np.linalg.norm(corr_real - corr_synth, ord='fro')
    print(f"{name:.<40} Frobenius Hata Puanı: {fark_normu:.4f}")
    return fark_normu

print("=== 1. Adversarial AUC ===")
hata_A = adversarial_validation(hedef_veri, df_sentetik_A, "Aday A (Ledoit-Wolf Gaussian)")
hata_B = adversarial_validation(hedef_veri, df_sentetik_B, "Aday B (Gaussian Copula)")
hata_C = adversarial_validation(hedef_veri, df_sentetik_C, "Aday C (CTGAN)")

print("\n=== 2. Frobenius Norm ===")
frob_A = frobenius_norm_test(hedef_veri, df_sentetik_A, "Aday A (Ledoit-Wolf Gaussian)")
frob_B = frobenius_norm_test(hedef_veri, df_sentetik_B, "Aday B (Gaussian Copula)")
frob_C = frobenius_norm_test(hedef_veri, df_sentetik_C, "Aday C (CTGAN)")

puanlar = {
    "Aday A (Ledoit-Wolf Gaussian)": hata_A + frob_A,
    "Aday B (Gaussian Copula)": hata_B + frob_B,
    "Aday C (CTGAN)": hata_C + frob_C
}

sampiyon_isim = min(puanlar, key=puanlar.get)
print("\n=== HAM (FİLTRESİZ) EN İYİ MODEL ===")
for algoritma, puan in sorted(puanlar.items(), key=lambda item: item[1]):
    print(f"{algoritma:.<40} Toplam Hata Puanı: {puan:.4f}")

print(f"\n HAM EN İYİ MODEL : {sampiyon_isim.upper()}")

=== 1. Adversarial AUC ===
Aday A (Ledoit-Wolf Gaussian)........... AUC Skoru: 1.0000 | Mükemmelliğe Uzaklık: 0.5000
Aday B (Gaussian Copula)................ AUC Skoru: 0.8467 | Mükemmelliğe Uzaklık: 0.3467
Aday C (CTGAN).......................... AUC Skoru: 0.9911 | Mükemmelliğe Uzaklık: 0.4911

=== 2. Frobenius Norm ===
Aday A (Ledoit-Wolf Gaussian)........... Frobenius Hata Puanı: 3.6586
Aday B (Gaussian Copula)................ Frobenius Hata Puanı: 2.5264
Aday C (CTGAN).......................... Frobenius Hata Puanı: 4.0597

=== HAM (FİLTRESİZ) EN İYİ MODEL ===
Aday B (Gaussian Copula)................ Toplam Hata Puanı: 2.8731
Aday A (Ledoit-Wolf Gaussian)........... Toplam Hata Puanı: 4.1586
Aday C (CTGAN).......................... Toplam Hata Puanı: 4.5508

 HAM EN İYİ MODEL : ADAY B (GAUSSIAN COPULA)


## 5. Adversarial Filtering ile Seçim
Ham modeller arasında en yüksek potansiyeli sergileyen Gaussian Copula modelinden 1000 adet geniş havuzlu örneklem üretilmiştir. Bu geniş havuz, gerçek verilerle birlikte bir "Düşman Sınıflandırıcıya" (Adversarial Classifier) verilmiştir. Modelin "sahte olduğundan emin olduğu" veriler elenmiş, "Gerçeğe en çok benzeyen (0.50 eşiğine en yakın)" 100 elit örnek nihai sentetik set olarak seçilmiştir.

In [9]:
df_adaylar = copula_model.sample(1000)
df_adaylar = fiziksel_sinirlara_hapset(df_adaylar)

X_filtre = pd.concat([hedef_veri, df_adaylar]).reset_index(drop=True)
y_filtre = np.array([0] * len(hedef_veri) + [1] * len(df_adaylar))

rf_dedektif = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
sahte_olasiliklari = cross_val_predict(rf_dedektif, X_filtre, y_filtre, cv=5, method='predict_proba')[:, 1]
sentetik_olasiliklari = sahte_olasiliklari[len(hedef_veri):]

df_adaylar['hata_payi'] = np.abs(sentetik_olasiliklari - 0.50)

df_sentetik_B = df_adaylar.sort_values(by='hata_payi').head(100).drop(columns=['hata_payi']).reset_index(drop=True)

## 6. Circular Evaluation Arındırılmış  Kalite Kontrolü
Aday B, "Adversarial Filtre" kullanılarak seçildiği için, nihai başarasını tekrar Adversarial AUC ile ölçmek akademik etiğe aykırıdır (Kendi sınavını kendi hazırlayan öğrenci yanılgısı). Bu sebeple şampiyon seçimi iki **tamamen bağımsız** test ile yapılmıştır:
1. **Frobenius Normu:** Matris yapısal bütünlüğü ve korelasyon sadakati.
2. **Anderson-Darling (AD) 1D Uzaklık Testi:** Değişkenlerin ekstrem uçlarının (kuyruk dağılımlarının) gerçek veriyle eşleşme kalitesi. AD p-değerinin >0.05 olması, ekstrem yangın şartlarının başarıyla taklit edildiğini kanıtlar.

In [10]:
def anderson_1d_test(real_data, synthetic_data, name):
    syn_sample = synthetic_data.head(len(real_data))
    ad_p_degerleri = []
    
    for col in real_data.columns:
        _, _, ad_p = anderson_ksamp([real_data[col].values, syn_sample[col].values])
        ad_p_degerleri.append(ad_p)
    
    ortalama_p = np.mean(ad_p_degerleri)
    print(f"{name:.<40} Ortalama AD p-değeri: {ortalama_p:.4f} (Beklenti: Yüksek)")
    return ortalama_p

print("=== 1. Frobenius Norm (Matris Yapı Testi) ===")
print("(Bu test seçilme filtresinden BAĞIMSIZDIR ve yapısal kaliteyi ölçer)\n")
frob_A_final = frobenius_norm_test(hedef_veri, df_sentetik_A, "Aday A (Ledoit-Wolf Gaussian)")
frob_B_final = frobenius_norm_test(hedef_veri, df_sentetik_B, "Aday B (FİLTRELENMİŞ Copula)")
frob_C_final = frobenius_norm_test(hedef_veri, df_sentetik_C, "Aday C (CTGAN)")

print("\n=== 2. Anderson-Darling 1D Uzaklık (Kuyruk Testi) ===")
print("(Bu test de BAĞIMSIZDIR. Değer yüksekse ekstrem olay kuyrukları uyumludur.)\n")
ad_A_final = anderson_1d_test(hedef_veri, df_sentetik_A, "Aday A (Ledoit-Wolf Gaussian)")
ad_B_final = anderson_1d_test(hedef_veri, df_sentetik_B, "Aday B (FİLTRELENMİŞ Copula)")
ad_C_final = anderson_1d_test(hedef_veri, df_sentetik_C, "Aday C (CTGAN)")

# Şampiyon seçimi sadece bağımsız test olan Frobenius normuna göre yapılıyor.
final_puanlar = {
    "Aday A (Ledoit-Wolf Gaussian)": frob_A_final,
    "Aday B (Filtrelenmiş Copula)": frob_B_final,
    "Aday C (CTGAN)": frob_C_final
}

sampiyon_isim_final = min(final_puanlar, key=final_puanlar.get)

print("\n=== FİNAL KARAR (En Düşük Frobenius Hata Normu) ===")
for algoritma, puan in sorted(final_puanlar.items(), key=lambda item: item[1]):
    print(f"{algoritma:.<40} Fiziksel Hata Puanı: {puan:.4f}")

print(f"\n EN İYİ MODEL: {sampiyon_isim_final.upper()} ")

=== 1. Frobenius Norm (Matris Yapı Testi) ===
(Bu test seçilme filtresinden BAĞIMSIZDIR ve yapısal kaliteyi ölçer)

Aday A (Ledoit-Wolf Gaussian)........... Frobenius Hata Puanı: 3.6586
Aday B (FİLTRELENMİŞ Copula)............ Frobenius Hata Puanı: 1.5180
Aday C (CTGAN).......................... Frobenius Hata Puanı: 4.0597

=== 2. Anderson-Darling 1D Uzaklık (Kuyruk Testi) ===
(Bu test de BAĞIMSIZDIR. Değer yüksekse ekstrem olay kuyrukları uyumludur.)

Aday A (Ledoit-Wolf Gaussian)........... Ortalama AD p-değeri: 0.0283 (Beklenti: Yüksek)
Aday B (FİLTRELENMİŞ Copula)............ Ortalama AD p-değeri: 0.1010 (Beklenti: Yüksek)
Aday C (CTGAN).......................... Ortalama AD p-değeri: 0.0450 (Beklenti: Yüksek)

=== FİNAL KARAR (En Düşük Frobenius Hata Normu) ===
Aday B (Filtrelenmiş Copula)............ Fiziksel Hata Puanı: 1.5180
Aday A (Ledoit-Wolf Gaussian)........... Fiziksel Hata Puanı: 3.6586
Aday C (CTGAN).......................... Fiziksel Hata Puanı: 4.0597

 EN İYİ MODEL:

In [11]:
import os

df_sentetik_B['rain'] = 0

gun_verileri = elit_mega_yanginlar[['day_sin', 'day_cos', 'is_weekend']].sample(n=len(df_sentetik_B), replace=True, random_state=42).reset_index(drop=True)
df_sentetik_B['day_sin'] = gun_verileri['day_sin']
df_sentetik_B['day_cos'] = gun_verileri['day_cos']
df_sentetik_B['is_weekend'] = gun_verileri['is_weekend']

df_sentetik_B['is_peak_season'] = ((df_sentetik_B['temp'] > 22) & (df_sentetik_B['DC'] > 500)).astype(int)

df_sentetik_B['distance_to_center'] = np.sqrt((df_sentetik_B['X'] - 5)**2 + (df_sentetik_B['Y'] - 5)**2)
df_sentetik_B['dynamic_seasonal_hotspot_distance'] = np.where(
    df_sentetik_B['is_peak_season'] == 1,
    np.sqrt((df_sentetik_B['X'] - 8)**2 + (df_sentetik_B['Y'] - 6)**2),
    np.sqrt((df_sentetik_B['X'] - 6)**2 + (df_sentetik_B['Y'] - 5)**2)
)

df_sentetik_B['moderate_wind_danger'] = ((df_sentetik_B['wind'] >= 3.5) & (df_sentetik_B['wind'] <= 6.0)).astype(int)
df_sentetik_B['hot_and_dry'] = ((df_sentetik_B['temp'] >= 20.0) & (df_sentetik_B['RH'] <= 40)).astype(int)
df_sentetik_B['double_drought'] = ((df_sentetik_B['FFMC'] >= 88.0) & (df_sentetik_B['DC'] >= 500.0)).astype(int)
df_sentetik_B['ISI_x_DC'] = df_sentetik_B['ISI'] * df_sentetik_B['DC']

# Kolonları orijinal veriye göre sıralıyoruz
df_sentetik_B = df_sentetik_B[df_train.columns]

# Orijinal Eğitim verisi ile Sentetik veriyi birleştiriyoruz
df_train_augmented = pd.concat([df_train, df_sentetik_B]).reset_index(drop=True)

# HEDEF DEĞİŞKEN SIZINTISINI (Target Leakage) BURADA DA ÖNLÜYORUZ!
# area değişkenini X matrisinden koparıp y olarak ayrı kaydediyoruz.
y_train_augmented_v2 = df_train_augmented['area']
X_train_augmented_v2 = df_train_augmented.drop(columns=['area'])

save_dir = '../../data/processed'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

X_train_augmented_v2.to_csv(f'{save_dir}/X_train_augmented_v2.csv', index=False)
y_train_augmented_v2.to_csv(f'{save_dir}/y_train_augmented_v2.csv', index=False)

print(f"Toplam Çoğaltılmış Gözlem Sayısı: {len(X_train_augmented_v2)}")
print("Kayıt Başarılı! Çoğaltılmış veri setinde de X ve y ayrılarak sızıntı engellendi.")

Toplam Çoğaltılmış Gözlem Sayısı: 513
Kayıt Başarılı! Çoğaltılmış veri setinde de X ve y ayrılarak sızıntı engellendi.
